# Projeto 02: Agente Farmacêutico Explicador de Bulas

## Carregamento de bibliotecas

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv
import random
import os

In [2]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("Chave carregada:", api_key[:5] + "...")

Chave carregada: sk-or...


## Carregamento dos documentos

In [3]:
caminhos_bulas = [
    './data/dipirona.pdf',
    './data/paracetamol.pdf'
]

documentos = []

for caminho in caminhos_bulas:
    loader = PyPDFLoader(caminho)
    docs = loader.load()
    for doc in docs:
        doc.metadata['medicamento'] = caminho.split('/')[-1].replace('.pdf', '')

    documentos.extend(docs)

len(documentos)

30

## Extração, Limpeza e Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documentos)

len(chunks)

271

## Enriquecimento de metadados

In [6]:
for chunk in chunks:

    texto = chunk.page_content.lower()

    if 'identificação de medicamento' in texto or 'composição' in texto:
        chunk.metadata['categoria'] = 'identificacao'

    elif 'indicação' in texto or 'para que este medicamento é indicado' in texto:
        chunk.metadata['categoria'] = 'indicacao'

    elif 'como este medicamento funciona' in texto or 'ação' in texto:
        chunk.metadata['categoria'] = 'como_funciona'

    elif 'contraindicação' in texto or 'quando não devo usar' in texto:
        chunk.metadata['categoria'] = 'contraindicacao'

    elif 'advertência' in texto or 'precaução' in texto or 'o que devo saber antes de usar' in texto:
        chunk.metadata['categoria'] = 'advertencias_precaucoes'

    elif 'interação' in texto or 'interações medicamentosas' in texto:
        chunk.metadata['categoria'] = 'interações'

    elif 'dose' in texto or 'posologia' in texto or 'como devo usar' in texto:
        chunk.metadata['categoria'] = 'posologia'

    elif 'reações adversas' in texto or 'quais os males' in texto:
        chunk.metadata['categoria'] = 'reacoes_adversas'

    elif 'onde, como e por quanto tempo posso guardar' in texto or 'armazenar' in texto:
        chunk.metadata['categoria'] = 'armazenamento'

    elif 'quantidade maior do que a indicada' in texto or 'superdosagem' in texto:
        chunk.metadata['categoria'] = 'superdosagem'

    else:
        chunk.metadata['categoria'] = 'geral'

In [9]:
# retorno de chunks aleatórias
chunks_aleatorios = random.sample(chunks, 2)

for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(f"\n-------- Chunk Aleatório {1} --------")
    print(f"Metadados: {chunk.metadata}")
    print(f"Conteúdo (início):\n{chunk.page_content[:300]}")
    


-------- Chunk Aleatório 1 --------
Metadados: {'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2013-10-22T16:50:37-02:00', 'author': 'mctrlarissa', 'moddate': '2013-10-22T16:50:37-02:00', 'title': 'Microsoft Word - Anexo 01 - Modelo Folha de Rosto_Bula_Bio_sol or', 'source': './data/dipirona.pdf', 'total_pages': 20, 'page': 18, 'page_label': '19', 'medicamento': 'dipirona', 'categoria': 'advertencias_precaucoes'}
Conteúdo (início):
Dipirona monoidratada solução oral (gotas)_BU 02_VPS  8 
ou Necrólise Epidérmica Tóxica (síndrom e bolhosa rara e grave, caracterizada 
clinicamente por necrose em grandes áreas da epiderme. Confere ao paciente aspecto 
de grande queimadura) (vide Advertências). Deve-se interromper imediatamente o u

-------- Chunk Aleatório 1 --------
Metadados: {'producer': 'Microsoft® Word para Microsoft 365', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2022-05-02T09:18:51-03:00', 'author':

## Geração de Embeddings e Banco Vetorial

In [10]:
# criação dos embeddings
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_bulas'
)

## Recuperação de Contexto (Retriever)

In [11]:
retriever = vectorstore.as_retriever(
    search_kwargs={'k': 4}
)

## Integração com Pipeline RAG

In [14]:
llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'De acordo com o seu conhecimento e utilizando o contexto disponibilizado, responda a seguinte pergunta.'),
        ('human', "Pergunta: {question}\n\nContexto:\n{context}")
    ]
)

# construção da pipeline
chain  = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x['question']
    }
    | prompt
    |llm
)

full_chain = RunnableParallel(
    answer=chain,
    context = lambda x: retriever.invoke(x['question'])
)

## Teste e Validação das Respostas

In [18]:
pergunta = "Quais os males que dipirona pode causar?"

response = full_chain.invoke({"question": pergunta})

print(f"Pergunta: {pergunta}")
print(f"\nResposta da LLM: {response['answer'].content}")

print("\nDocumentos Utilizados como contexto:")
idx = 1
for doc in response["context"]:
    print(f"{idx}) {doc.page_content[:300]}\n****Fonte:{doc.metadata.get("source", "Documento desconhecimento")}\n****Página: {doc.metadata.get('page', 'N/A')}")
    idx += 1

Pergunta: Quais os males que dipirona pode causar?

Resposta da LLM: Decalculação de respostas a partir do contexto fornecido sugere que os possíveis riscos da dipirona incluem:

a) Reações adversas graves e potencialmente fatais, incluindo anafilaxia.
b) Problemas hepáticos e renais, especialmente em pacientes com insuficiência hepática ou renal.
c) Reações hipotensivas isoladas.
d) Aumento do risco de complicações em pacientes com insuficiência renal ou hepática, levando a condições graves e potencialmente fatais.
e) Contraindicações também se aplicam a mulheres grávidas e lactantes, e o medicamento só deve ser utilizado sob orientação médica nesses casos.

É importante observar que outras informações específicas sobre o uso e as precauções que devem ser tomadas ao usar a dipirona podem variar, dependendo da gravidade da condição da doença e dos resultados de testes e dos casos específicos do paciente.

Documentos Utilizados como contexto:
1) funções hepática e renal estarem prejudic